# Savant full - two protein-embedding variants

Trains **four** models: an XGBoost pKd regressor and an XGBoost binder
classifier, each in two variants that differ **only** in the protein embedding.

| variant | protein block | width | source |
|---|---|---|---|
| `esm650` | ESM2-650M, mean-pooled | 1280 | `facebook/esm2_t33_650M_UR50D` |
| `esm8`   | ESM2-8M, mean-pooled   | 320  | `facebook/esm2_t6_8M_UR50D` |

Everything else is identical: same rows, same gate, same split, same ligand
features. So a difference in the held-out numbers is attributable to the protein
representation and nothing else - which is the point of running both.

**Feature block order (load-bearing).** The width check cannot catch a wrong
order, because two orders can share a width. This is the order
`train_full_dataset.py` used and the one `savant05.py`'s `full` model expects:

    ChemBERTa ligand embedding   384
    RDKit named descriptors      226   (columns saved to rdkit_columns.json)
    ECFP4 count fingerprint     1024
    ESM2 protein embedding      1280 | 320
    AA composition + length       21
                                ----
                          esm650 2935     esm8 1975

The `esm650` variant is therefore drop-in compatible with
`savant05.py --model full`.

## What this notebook reuses

The build stages are imported from `train_full_dataset.py` rather than copied,
so there is one implementation of the RDKit/ChemBERTa/ESM encoding and it cannot
drift. What is new here: the two-variant protein stage, hyperparameter tuning,
and the Random Forest classifier.

## One thing to read before running

**The benchmark gate targets `savant_benchmark_v3.csv`.** `train_full_dataset.py`
defaults to the older 1000-pair file. Gating against the wrong benchmark leaves
near-duplicate ligands in training and turns scoring into a memorisation test.

## Why both models are XGBoost

Beyond matching the regressor, it buys three things at this scale:

- **The binned matrices are built once.** `QuantileDMatrix` binning does not
  depend on the label, so the fit/val matrices are relabelled from pKd to
  binder and reused - no second ~14 GB build per variant.
- **No row subsampling.** Both models see all 1.16M fit rows. A Random Forest
  here would need the fit set cut to ~400k to stay inside memory, which would
  have made the classifier a weaker test of the embedding than the regressor.
- **Portable bundles.** Both models save as `save_model()` JSON, which carries
  its own column list and intercept and loads across XGBoost versions. Nothing
  in the bundle is a scikit-learn pickle, so nothing carries a sklearn version
  requirement.

In [ ]:
import json, os, sys, time
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()                       # run this notebook from ML_model/
sys.path.insert(0, str(HERE))
import train_full_dataset as tfd        # the proven build stages

INPUT_CSV = HERE / "Input" / "input.csv" if (HERE / "Input").exists() \
            else HERE.parent / "Input" / "input.csv"
BENCH_CSV = HERE / "benchmark" / "output" / "savant_benchmark_v3.csv"
OUT       = HERE / "work_full_v2"       # shared feature store + models

## the two variants; only the protein encoder differs
VARIANTS = {
    "esm650": {"model": "facebook/esm2_t33_650M_UR50D", "file": "protein_esm2_650M.npy",
               "batch": 8,  "dim": 1280},
    "esm8":   {"model": "facebook/esm2_t6_8M_UR50D",    "file": "protein_esm2_8M.npy",
               "batch": 64, "dim": 320},
}

SEED        = 1337
BINDER_PKD  = 6.0
GATE_CUTOFF = 0.50
MIN_PKD, MAX_PKD = 2.0, 12.0
N_THREADS   = -1
RDKIT_WORKERS = os.cpu_count()

## tuning budget
TUNE_ROWS  = 200_000       # rows sampled from fit for the search
TUNE_ITERS = 12            # random-search draws per model

OUT.mkdir(parents=True, exist_ok=True)
log = tfd.log

assert INPUT_CSV.exists(), f"missing {INPUT_CSV}"
if not BENCH_CSV.exists():
    raise SystemExit(
        f"missing {BENCH_CSV}.\n"
        "The gate needs the v3 benchmark. Copy savant_benchmark_v3.csv into\n"
        "benchmark/output/ before running - gating against the older 1000-pair\n"
        "file would leave v3 benchmark ligands in training."
    )
log(f"input {INPUT_CSV}")
log(f"gate  {BENCH_CSV}")
log(f"out   {OUT}")

## 1-2. Rows and the benchmark gate

Every stage writes its result into `OUT` and is skipped if that file already
exists, so a walltime kill resumes rather than restarts. Delete a file to force
that stage to rerun.

The gate drops every training ligand within ECFP4 Tanimoto 0.50 of any v3
benchmark ligand. It is the slowest non-GPU stage (~1M unique ligands screened
against ~370 benchmark ligands) and runs once for both variants.

In [ ]:
if not (OUT / "dataset_rows.csv").exists():
    df = tfd.load_rows(INPUT_CSV, MIN_PKD, MAX_PKD)
    df = tfd.gate_benchmark(df, BENCH_CSV, GATE_CUTOFF)

    ligands  = pd.Index(df.smiles.unique())
    proteins = pd.Index(df.seq.unique())
    log(f"unique ligands {len(ligands)}, unique proteins {len(proteins)}")

    rows = pd.DataFrame({
        "ligand_id":  ligands.get_indexer(df.smiles),
        "protein_id": proteins.get_indexer(df.seq),
        "neg_log10_affinity_M": df.neg_log10_affinity_M.astype(np.float32),
    })
    rows["is_binder"] = (rows.neg_log10_affinity_M > BINDER_PKD).astype(np.int8)
    rows.to_csv(OUT / "dataset_rows.csv", index=False)
    pd.DataFrame({"ligand_id": range(len(ligands)), "smiles": ligands}
                 ).to_csv(OUT / "ligands.csv", index=False)
    pd.DataFrame({"protein_id": range(len(proteins)), "seq": proteins}
                 ).to_csv(OUT / "proteins.csv", index=False)
    log(f"wrote dataset_rows.csv ({len(rows)} rows)")
else:
    log("dataset_rows.csv exists, reusing")

rows_all = pd.read_csv(OUT / "dataset_rows.csv")
ligands  = pd.read_csv(OUT / "ligands.csv").smiles.tolist()
proteins = pd.read_csv(OUT / "proteins.csv").seq.tolist()
print(f"{len(rows_all):,} rows | {len(ligands):,} ligands | {len(proteins):,} proteins")
print(f"binder rate at pKd > {BINDER_PKD}: {rows_all.is_binder.mean():.3f}")

## 3. Ligand features - shared by both variants

RDKit descriptors, the 11 AddHs variants, ECFP4 counts, and the ChemBERTa
embedding. These depend only on the ligand, so they are computed once and both
variants read the same arrays. This is most of the CPU cost of the build
(~84 ligands/sec/core), which is why it is worth not doing it twice.

In [ ]:
if not (OUT / "ligand_rdkit.npy").exists():
    desc, ecfp = tfd.ligand_tables(ligands, RDKIT_WORKERS)
    np.save(OUT / "ligand_rdkit.npy", desc.to_numpy(np.float32))
    np.save(OUT / "ligand_ecfp.npy", ecfp)
    json.dump(list(desc.columns), open(OUT / "rdkit_columns.json", "w"), indent=1)
    log(f"rdkit {desc.shape[1]} columns, ecfp {ecfp.shape[1]}")
else:
    log("cached: ligand_rdkit.npy / ligand_ecfp.npy")

if not (OUT / "ligand_chemberta.npy").exists():
    np.save(OUT / "ligand_chemberta.npy",
            tfd.encode(ligands, tfd.CHEMBERTA, 256, 256))
else:
    log("cached: ligand_chemberta.npy")

if not (OUT / "protein_aa.npy").exists():
    np.save(OUT / "protein_aa.npy", tfd.aa_composition(proteins))
else:
    log("cached: protein_aa.npy")

for f in ("ligand_chemberta.npy", "ligand_rdkit.npy", "ligand_ecfp.npy", "protein_aa.npy"):
    print(f"  {f:26s} {np.load(OUT / f, mmap_mode='r').shape}")

## 4. Protein embeddings - one per variant

The only stage that differs. ESM2-8M is ~80x smaller than 650M, so it encodes in
minutes rather than hours and fits a much larger batch.

In [ ]:
for name, cfg in VARIANTS.items():
    path = OUT / cfg["file"]
    if path.exists():
        log(f"cached: {cfg['file']} {np.load(path, mmap_mode='r').shape}")
        continue
    log(f"encoding {len(proteins)} proteins with {cfg['model']}")
    emb = tfd.encode(proteins, cfg["model"], cfg["batch"], 1024)
    assert emb.shape[1] == cfg["dim"], \
        f"{name}: expected {cfg['dim']} dims, got {emb.shape[1]}"
    np.save(path, emb)
    log(f"wrote {cfg['file']} {emb.shape}")

## 5. The split

Protein-grouped, so no protein appears in both training and test. The inner
split used for early stopping holds out the **same key** as the outer one -
using a random inner split there lets the model tune against proteins it will be
tested on, which inflates the early-stopping point.

Rows are reordered to `fit | val | test` *before* the matrix is built, so each
split is a contiguous slice - a view, not a copy. Fancy-indexing the matrix three
times instead would add another ~14 GB per variant.

The split is computed once and reused by both variants, so the two are compared
on identical rows.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

y_all  = rows_all.neg_log10_affinity_M.to_numpy(np.float32)
groups = rows_all.protein_id.to_numpy()

tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
              .split(np.zeros(len(rows_all)), y_all, groups))
i_fit, i_val = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED)
                    .split(tr, y_all[tr], groups=groups[tr]))
fit, val = tr[i_fit], tr[i_val]

N_FIT, N_VAL, N_TE = len(fit), len(val), len(te)
CUT = N_FIT + N_VAL

rows = rows_all.iloc[np.concatenate([fit, val, te])].reset_index(drop=True)
y      = rows.neg_log10_affinity_M.to_numpy(np.float32)
binder = rows.is_binder.to_numpy(np.int8)

assert not (set(groups[fit]) & set(groups[te])), "protein leaked fit -> test"
assert not (set(groups[val]) & set(groups[te])), "protein leaked val -> test"
print(f"fit {N_FIT:,}  val {N_VAL:,}  test {N_TE:,}")
print(f"proteins: fit {len(set(groups[fit])):,}  val {len(set(groups[val])):,}  "
      f"test {len(set(groups[te])):,}  (disjoint)")
print(f"test binder rate {binder[CUT:].mean():.3f}")

## Assembling one variant's matrix

Same chunked memmap gather as `train_full_dataset.assemble`, but parameterised by
which protein embedding file to use. The per-entity blocks stay on disk and rows
are gathered a chunk at a time straight into the preallocated output, so the
matrix is never held twice.

In [ ]:
def assemble_variant(rows, esm_file, chunk=200_000):
    '''(n, width) float32 in the training block order for one variant.'''
    lparts = [np.load(OUT / f, mmap_mode="r") for f in
              ("ligand_chemberta.npy", "ligand_rdkit.npy", "ligand_ecfp.npy")]
    pparts = [np.load(OUT / f, mmap_mode="r") for f in (esm_file, "protein_aa.npy")]
    width  = sum(p.shape[1] for p in lparts + pparts)

    X = np.empty((len(rows), width), dtype=np.float32)
    log(f"matrix {X.shape} ({X.nbytes / 1e9:.1f} GB) from {esm_file}")
    li, pi = rows.ligand_id.to_numpy(), rows.protein_id.to_numpy()
    for start in range(0, len(rows), chunk):
        stop, col = min(start + chunk, len(rows)), 0
        for part, ids in ((lparts, li), (pparts, pi)):
            for block in part:
                X[start:stop, col:col + block.shape[1]] = block[ids[start:stop]]
                col += block.shape[1]
    return X

for name, cfg in VARIANTS.items():
    w = 384 + np.load(OUT / "ligand_rdkit.npy", mmap_mode="r").shape[1] + 1024 + cfg["dim"] + 21
    cfg["width"] = w
    print(f"  {name:8s} {w} features  (~{len(rows) * w * 4 / 1e9:.1f} GB for all rows)")

## 6. Hyperparameter tuning

Random search, not a grid: with six interacting parameters a grid spends most of
its budget on combinations that differ in a dimension that does not matter.

Two deliberate choices:

- **Searched on a subsample** (`TUNE_ROWS`), scored on the **full** validation
  set. Tuning on all 1.16M rows would cost more than the training run it is meant
  to configure, and the ranking of parameter sets is stable well below full size.
- **A held-out val set, not cross-validation.** CV would refit k times per draw,
  and re-splitting inside the fit set risks breaking the protein grouping.

Search on the `esm650` variant and reuse the parameters for `esm8`. Tuning each
separately would confound the comparison: a difference in the results could then
come from the search landing better for one variant rather than from the
embedding itself.

Both searches share one pair of binned matrices - the classifier search relabels
them rather than rebuilding, since the binning does not depend on the label.

**Resuming.** A trial costs 3-15 minutes, so a full search is over an hour and
losing one to a walltime kill is expensive. Two safeguards:

- `best_params_reg.json` and `best_params_clf.json` are cached and checked
  **independently**. If the regressor search finished and the classifier did not,
  rerunning skips the regressor.
- `tuning_trials_reg.csv` / `tuning_trials_clf.csv` are rewritten after **every**
  trial. If a search dies partway, the completed trials are still on disk - pick
  the best row by hand and write the params file yourself to skip that search.

In [ ]:
import xgboost as xgb

rng = np.random.default_rng(SEED)
SPACE = {
    "max_depth":        [6, 7, 8, 9, 10, 12],
    "eta":              [0.03, 0.05, 0.08, 0.12],
    "subsample":        [0.6, 0.7, 0.8, 0.9],
    "colsample_bytree": [0.3, 0.4, 0.5, 0.7],
    "min_child_weight": [1, 3, 5, 10, 20],
    "reg_lambda":       [0.5, 1.0, 2.0, 5.0],
}

reg_file, clf_file = OUT / "best_params_reg.json", OUT / "best_params_clf.json"

def _py(v):
    '''numpy scalar -> python scalar. pandas .loc returns np.int64/np.float64,
    and json.dumps refuses both ("Object of type int64 is not JSON serializable").'''
    return v.item() if hasattr(v, "item") else v

def _save_params(t, path):
    best = {k: _py(t.loc[0, k]) for k in SPACE}
    path.write_text(json.dumps(best, indent=1))
    log(f"wrote {path.name}: {best}")
    return best

## Cached per model, not as a pair: each search costs over an hour, so a failure
## in the second one must not throw away the first.
BEST_REG = json.loads(reg_file.read_text()) if reg_file.exists() else None
BEST_CLF = json.loads(clf_file.read_text()) if clf_file.exists() else None

if BEST_REG and BEST_CLF:
    log(f"reusing cached params\n  reg {BEST_REG}\n  clf {BEST_CLF}")
else:
    sub = rng.choice(N_FIT, size=min(TUNE_ROWS, N_FIT), replace=False)
    sub.sort()
    Xt = assemble_variant(rows.iloc[sub], VARIANTS["esm650"]["file"])
    Xv = assemble_variant(rows.iloc[N_FIT:CUT], VARIANTS["esm650"]["file"])
    dt = xgb.QuantileDMatrix(Xt, label=y[sub])
    dv = xgb.QuantileDMatrix(Xv, label=y[N_FIT:CUT], ref=dt)
    del Xt, Xv
    log(f"tuning on {len(sub):,} fit rows, scoring on all {N_VAL:,} val rows")

    def search(objective, metric, maximise, tag, extra=None):
        trials, csv = [], OUT / f"tuning_trials_{tag}.csv"
        for i in range(TUNE_ITERS):
            ## index rather than rng.choice: choice on a mixed int/float list
            ## upcasts everything to float, and xgboost rejects a float max_depth
            p = {k: v[rng.integers(len(v))] for k, v in SPACE.items()}
            b = xgb.train({**p, **(extra or {}), "objective": objective,
                           "eval_metric": metric, "tree_method": "hist",
                           "nthread": N_THREADS, "seed": SEED},
                          dt, num_boost_round=1500, evals=[(dv, "val")],
                          early_stopping_rounds=40, verbose_eval=False)
            trials.append({**p, metric: float(b.best_score),
                           "best_iter": int(b.best_iteration)})
            ## rewritten every trial: each one costs 3-15 min, so a walltime kill
            ## or a crash must not discard the ones already finished
            pd.DataFrame(trials).to_csv(csv, index=False)
            log(f"  [{i+1}/{TUNE_ITERS}] {metric} {b.best_score:.4f} "
                f"@{b.best_iteration}  {p}")
        t = pd.DataFrame(trials).sort_values(metric, ascending=not maximise)
        return t.reset_index(drop=True)

    if BEST_REG is None:
        log("--- regressor ---")
        t_reg = search("reg:squarederror", "rmse", maximise=False, tag="reg")
        display(t_reg)
        BEST_REG = _save_params(t_reg, reg_file)
    else:
        log(f"reusing cached regressor params: {BEST_REG}")

    if BEST_CLF is None:
        ## same binned matrices, relabelled to the binder target - the
        ## QuantileDMatrix binning does not depend on the label
        POS = int(binder[sub].sum())
        SPW = (len(sub) - POS) / max(POS, 1)
        dt.set_label(binder[sub])
        dv.set_label(binder[N_FIT:CUT])
        log(f"--- classifier --- (scale_pos_weight {SPW:.3f}, "
            f"binder rate {binder[sub].mean():.3f})")
        t_clf = search("binary:logistic", "aucpr", maximise=True, tag="clf",
                       extra={"scale_pos_weight": SPW})
        display(t_clf)
        BEST_CLF = _save_params(t_clf, clf_file)
    else:
        log(f"reusing cached classifier params: {BEST_CLF}")

    del dt, dv

## 7. Train both variants

One variant at a time, freeing its matrix before starting the next - the two
matrices together are ~24 GB.

Within a variant the fit/val matrices are binned once and relabelled for the
classifier, so each variant costs one matrix build, not two. Both models early-stop
against the val set.

Boosters are saved with `save_model()` JSON rather than pickled: a pickled
booster can come back with `base_score` reset to the 0.5 default, which shifts
every predicted pKd down by about 6. The JSON carries the fitted intercept.

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score,
                             roc_auc_score)
import joblib

POS_FIT = int(binder[:N_FIT].sum())
SPW_FIT = (N_FIT - POS_FIT) / max(POS_FIT, 1)
RESULTS, PRED = {}, {}

for name, cfg in VARIANTS.items():
    log(f"===== {name} ({cfg['width']} features) =====")
    X = assemble_variant(rows, cfg["file"])

    dfit = xgb.QuantileDMatrix(X[:N_FIT], label=y[:N_FIT])
    dval = xgb.QuantileDMatrix(X[N_FIT:CUT], label=y[N_FIT:CUT], ref=dfit)
    reg = xgb.train({**BEST_REG, "objective": "reg:squarederror", "eval_metric": "rmse",
                     "tree_method": "hist", "nthread": N_THREADS, "seed": SEED},
                    dfit, num_boost_round=3000, evals=[(dval, "val")],
                    early_stopping_rounds=50, verbose_eval=200)
    pred = reg.inplace_predict(X[CUT:])

    ## relabel the same binned matrices rather than building a second pair
    dfit.set_label(binder[:N_FIT])
    dval.set_label(binder[N_FIT:CUT])
    clf = xgb.train({**BEST_CLF, "objective": "binary:logistic", "eval_metric": "aucpr",
                     "tree_method": "hist", "nthread": N_THREADS, "seed": SEED,
                     "scale_pos_weight": SPW_FIT},
                    dfit, num_boost_round=3000, evals=[(dval, "val")],
                    early_stopping_rounds=50, verbose_eval=200)
    proba = clf.inplace_predict(X[CUT:])
    del dfit, dval

    yt, bt = y[CUT:], binder[CUT:]
    RESULTS[name] = {
        "variant": name, "n_features": int(cfg["width"]),
        "esm_model": cfg["model"], "esm_dim": cfg["dim"],
        "n_fit": N_FIT, "n_val": N_VAL, "n_test": N_TE,
        "reg_best_iteration": int(reg.best_iteration),
        "clf_best_iteration": int(clf.best_iteration),
        "RMSE": float(np.sqrt(mean_squared_error(yt, pred))),
        "MAE": float(mean_absolute_error(yt, pred)),
        "R2": float(r2_score(yt, pred)),
        "Pearson": float(np.corrcoef(yt, pred)[0, 1]),
        "Spearman": float(spearmanr(yt, pred).correlation),
        "pred_std_ratio": float(pred.std() / yt.std()),
        "Accuracy": float(accuracy_score(bt, proba >= 0.5)),
        "F1": float(f1_score(bt, proba >= 0.5, zero_division=0)),
        "ROC_AUC": float(roc_auc_score(bt, proba)),
        "PR_AUC": float(average_precision_score(bt, proba)),
    }
    PRED[name] = {"pred": pred, "proba": proba}
    for k, v in RESULTS[name].items():
        print(f"  {k:20s} {v}")

    vdir = OUT / name
    vdir.mkdir(exist_ok=True)
    reg.save_model(vdir / "xgb_regressor.json")
    clf.save_model(vdir / "xgb_classifier.json")
    json.dump(RESULTS[name], open(vdir / "metrics.json", "w"), indent=1)
    log(f"saved {vdir}/xgb_regressor.json, xgb_classifier.json, metrics.json")

    del X, reg, clf
    import gc; gc.collect()

## 8. Results

`pred_std_ratio` is the diagnostic to watch alongside the correlations: it is the
predicted spread over the true spread. Squared-error training on a noisy label
shrinks predictions toward the mean, and a model at 0.4 is making a much narrower
claim than the labels do. A high Pearson with a low ratio still ranks well but
will look badly calibrated in absolute pKd.

In [ ]:
res = pd.DataFrame(RESULTS).T
cols = ["n_features", "RMSE", "MAE", "R2", "Pearson", "Spearman", "pred_std_ratio",
        "ROC_AUC", "PR_AUC", "Accuracy", "F1"]
display(res[cols].astype(float).round(4))

res.to_csv(OUT / "variant_comparison.csv")
yt = y[CUT:]
print(f"\nconstant-mean baseline RMSE on this test set: {yt.std():.3f}")
print("a model above this line is not beating 'predict the average'")

In [ ]:
import matplotlib.pyplot as plt

metrics = [("Pearson", "higher better"), ("Spearman", "higher better"),
           ("R2", "higher better"), ("RMSE", "lower better"),
           ("ROC_AUC", "higher better"), ("PR_AUC", "higher better")]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
names = list(RESULTS)
for ax, (m, note) in zip(axes.ravel(), metrics):
    vals = [RESULTS[n][m] for n in names]
    ax.bar(names, vals, color=["#2a78d6", "#eb6834"], width=0.6)
    ax.set_title(f"{m}\n({note})", fontsize=10)
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.margins(y=0.18)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("ESM2-650M vs ESM2-8M - identical rows, split and ligand features", y=1.0)
fig.tight_layout()
fig.savefig(OUT / "variant_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Transferable bundles

One `.pkl` per variant holding both boosters as `save_model()` JSON bytes, plus
the RDKit column list, the block layout and the metrics. Nothing here is a
scikit-learn pickle, so the bundle carries no sklearn version requirement and
loads on any machine with a compatible XGBoost.

These are what `savant_v1/savant1.py` loads.

In [ ]:
for name, cfg in VARIANTS.items():
    vdir = OUT / name
    cols = json.loads((OUT / "rdkit_columns.json").read_text())
    bundle = {
        "variant": name,
        "esm_model": cfg["model"],
        "block_order": ["chemberta", "rdkit_named", "ecfp", "esm", "aa"],
        "block_widths": [384, len(cols), 1024, cfg["dim"], 21],
        "n_features": int(cfg["width"]),
        "rdkit_columns": cols,
        "regressor_json":  (vdir / "xgb_regressor.json").read_bytes(),
        "classifier_json": (vdir / "xgb_classifier.json").read_bytes(),
        "metrics": RESULTS[name],
        "xgboost_version": xgb.__version__,
        "binder_pkd": BINDER_PKD,
    }
    assert sum(bundle["block_widths"]) == bundle["n_features"]
    path = OUT / f"savant_full_{name}.pkl"
    joblib.dump(bundle, path, compress=3)
    print(f"{path.name}  {path.stat().st_size / 1e6:.0f} MB  "
          f"{bundle['n_features']} features  xgboost {bundle['xgboost_version']}")

print("\nblock order is load-bearing and the width check cannot catch a wrong one.")
print("savant1.py rebuilds features in exactly this order.")